In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/boransaksham/echonext-singlelead/feature_info.json
/kaggle/input/datasets/boransaksham/echonext-singlelead/Y_train.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/Y_test.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/T_A_test.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/leadII_val.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/Y_val.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/leadII_train.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/leadII_test.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/T_B_val.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/T_A_train.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/T_A_val.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/T_B_train.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/leadII_no_split.npy
/kaggle/input/datasets/boransaksham/echonext-singlelead/T_B_test.npy


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
Device name: Tesla T4


In [3]:
# Cell 1: load single-lead data (corrected Kaggle path)
import numpy as np, torch, torch.nn as nn, json, copy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, confusion_matrix

DATA = '/kaggle/input/datasets/boransaksham/echonext-singlelead'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

def load_split(split):
    X = np.load(f'{DATA}/leadII_{split}.npy').astype(np.float32)
    TB = np.load(f'{DATA}/T_B_{split}.npy').astype(np.float32)
    Y = np.load(f'{DATA}/Y_{split}.npy').astype(np.float32)
    return X, TB, Y

X_train, TB_train, Y_train = load_split('train')
X_val, TB_val, Y_val = load_split('val')
X_test, TB_test, Y_test = load_split('test')

def znorm(x):
    return (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-6)
X_train, X_val, X_test = znorm(X_train), znorm(X_val), znorm(X_test)

pos_rates = torch.tensor([0.1789, 0.1324, 0.2438, 0.5237])
pos_weights = ((1 - pos_rates) / pos_rates).to(device)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

Device: cuda
train: (72475, 2500) val: (4626, 2500) test: (5442, 2500)


In [4]:
# Cell 2: CNN model + loss function
import torch.nn.functional as F

class ResBlock1D(nn.Module):
    def __init__(self, ch_in, ch_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(ch_in, ch_out, kernel_size=7, stride=stride, padding=3)
        self.bn1 = nn.BatchNorm1d(ch_out)
        self.conv2 = nn.Conv1d(ch_out, ch_out, kernel_size=7, stride=1, padding=3)
        self.bn2 = nn.BatchNorm1d(ch_out)
        self.act = nn.GELU()
        self.shortcut = nn.Sequential()
        if stride != 1 or ch_in != ch_out:
            self.shortcut = nn.Sequential(nn.Conv1d(ch_in, ch_out, kernel_size=1, stride=stride), nn.BatchNorm1d(ch_out))
    def forward(self, x):
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.act(out)

class ResNet1D(nn.Module):
    def __init__(self, d_out=256):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.BatchNorm1d(32), nn.GELU())
        self.layer1 = ResBlock1D(32, 64, stride=2)
        self.layer2 = ResBlock1D(64, 128, stride=2)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, d_out, stride=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.pool(x).squeeze(-1)

class SingleLeadCNNModel(nn.Module):
    def __init__(self, cnn_backbone, tabular_dim, cnn_dim=256, hidden=128, n_targets=4):
        super().__init__()
        self.backbone = cnn_backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.1))
        self.head_trunk = nn.Sequential(nn.Linear(cnn_dim + 32, hidden), nn.ReLU(), nn.Dropout(0.2))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, ecg, tabular):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        fused = torch.cat([feats, tab], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

def masked_bce_loss(logits, targets, pos_weights):
    losses = []
    for i in range(targets.shape[1]):
        col_t, col_l = targets[:, i], logits[:, i]
        mask = ~torch.isnan(col_t)
        if mask.sum() == 0: continue
        losses.append(F.binary_cross_entropy_with_logits(col_l[mask], col_t[mask], pos_weight=pos_weights[i]))
    return torch.stack(losses).mean()

print("Models defined.")

Models defined.


In [5]:
# Cell 3: train two seeded CNNs, save checkpoints properly this time
def train_cnn(seed, epochs=12, patience=4):
    torch.manual_seed(seed)
    net = SingleLeadCNNModel(ResNet1D(d_out=256).to(device), tabular_dim=TB_train.shape[1]).to(device)
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(Y_train)),
                               batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(Y_val)),
                             batch_size=64, shuffle=False)
    optimizer = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, y in train_loader:
            ecg, tab, y = ecg.to(device), tab.to(device), y.to(device)
            optimizer.zero_grad()
            loss = masked_bce_loss(net(ecg, tab), y, pos_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, y in val_loader:
                logits = net(ecg.to(device), tab.to(device))
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[seed{seed}] Epoch {epoch}: val_HFrEF_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[seed{seed}] Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/cnn_seed{seed}.pt')
    print(f"[seed{seed}] Saved to /kaggle/working/cnn_seed{seed}.pt. Best val AUROC: {best_val_auroc:.4f}")
    return net

net_s1 = train_cnn(seed=1)
net_s2 = train_cnn(seed=2)

[seed1] Epoch 0: val_HFrEF_AUROC=0.8443
[seed1] Epoch 1: val_HFrEF_AUROC=0.8542
[seed1] Epoch 2: val_HFrEF_AUROC=0.8591
[seed1] Epoch 3: val_HFrEF_AUROC=0.8603
[seed1] Epoch 4: val_HFrEF_AUROC=0.8548
[seed1] Epoch 5: val_HFrEF_AUROC=0.8567
[seed1] Epoch 6: val_HFrEF_AUROC=0.8590
[seed1] Epoch 7: val_HFrEF_AUROC=0.8578
[seed1] Early stopping at epoch 7
[seed1] Saved to /kaggle/working/cnn_seed1.pt. Best val AUROC: 0.8603
[seed2] Epoch 0: val_HFrEF_AUROC=0.8501
[seed2] Epoch 1: val_HFrEF_AUROC=0.8449
[seed2] Epoch 2: val_HFrEF_AUROC=0.8590
[seed2] Epoch 3: val_HFrEF_AUROC=0.8641
[seed2] Epoch 4: val_HFrEF_AUROC=0.8590
[seed2] Epoch 5: val_HFrEF_AUROC=0.8569
[seed2] Epoch 6: val_HFrEF_AUROC=0.8595
[seed2] Epoch 7: val_HFrEF_AUROC=0.8592
[seed2] Early stopping at epoch 7
[seed2] Saved to /kaggle/working/cnn_seed2.pt. Best val AUROC: 0.8641


In [6]:
# Cell 4: ensemble both CNNs, final honest test evaluation
def get_test_probs(net):
    net.eval()
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(Y_test)), batch_size=64)
    all_logits, all_y = [], []
    with torch.no_grad():
        for ecg, tab, y in test_loader:
            logits = net(ecg.to(device), tab.to(device))
            all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

logits1, y_test = get_test_probs(net_s1)
logits2, _ = get_test_probs(net_s2)
probs1, probs2 = 1/(1+np.exp(-logits1)), 1/(1+np.exp(-logits2))
probs_ensemble = (probs1 + probs2) / 2

print("=== SEED 1 ALONE (test) ===")
m = ~np.isnan(y_test[:, 0])
print(f"  hfref AUROC={roc_auc_score(y_test[m,0], probs1[m,0]):.4f}")

print("\n=== SEED 2 ALONE (test) ===")
print(f"  hfref AUROC={roc_auc_score(y_test[m,0], probs2[m,0]):.4f}")

print("\n=== ENSEMBLE (average of both, test) ===")
for i, name in enumerate(['hfref', 'rv_dysf', 'lvh', 'shd']):
    mm = ~np.isnan(y_test[:, i])
    yt, yp = y_test[mm, i], probs_ensemble[mm, i]
    auroc = roc_auc_score(yt, yp)
    auprc = average_precision_score(yt, yp)
    pred = (yp > 0.5).astype(int)
    acc = accuracy_score(yt, pred)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"  [{name}] AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")

=== SEED 1 ALONE (test) ===
  hfref AUROC=0.8602

=== SEED 2 ALONE (test) ===
  hfref AUROC=0.8618

=== ENSEMBLE (average of both, test) ===
  [hfref] AUROC=0.8649 AUPRC=0.5260 acc=0.8123 sens=0.7295 spec=0.8240
  [rv_dysf] AUROC=0.8412 AUPRC=0.3503 acc=0.8025 sens=0.7064 spec=0.8105
  [lvh] AUROC=0.7185 AUPRC=0.3688 acc=0.6358 sens=0.7172 spec=0.6161
  [shd] AUROC=0.7966 AUPRC=0.7561 acc=0.7288 sens=0.6613 spec=0.7788


In [8]:
# Cell 5: CNN-Transformer hybrid backbone
class CNNTransformerBackbone(nn.Module):
    def __init__(self, d_model=128, n_heads=4, n_layers=3, out_dim=256):
        super().__init__()
        self.cnn_stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=7, stride=4, padding=3), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, 500, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                            dropout=0.15, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out_proj = nn.Linear(d_model, out_dim)

    def forward(self, x):
        feats = self.cnn_stem(x.unsqueeze(1))
        feats = feats.transpose(1, 2)
        T = feats.shape[1]
        feats = feats + self.pos_embed[:, :T, :]
        out = self.transformer(feats)
        pooled = out.mean(dim=1)
        return self.out_proj(pooled)

# shape check
test_backbone = CNNTransformerBackbone().to(device)
with torch.no_grad():
    test_out = test_backbone(torch.randn(4, 2500).to(device))
print("Hybrid output shape:", test_out.shape)
print(f"Hybrid params: {sum(p.numel() for p in test_backbone.parameters())/1e6:.2f}M")

Hybrid output shape: torch.Size([4, 256])
Hybrid params: 0.77M


In [9]:
# Cell 6: hybrid fusion model + training (with more patience, since it needs a bit longer to find its footing)
class SingleLeadHybridModel(nn.Module):
    def __init__(self, backbone, tabular_dim, feat_dim=256, hidden=128, n_targets=4):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.1))
        self.head_trunk = nn.Sequential(nn.Linear(feat_dim + 32, hidden), nn.ReLU(), nn.Dropout(0.2))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, ecg, tabular):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        fused = torch.cat([feats, tab], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

def train_hybrid(seed=1, epochs=15, patience=5):
    torch.manual_seed(seed)
    net = SingleLeadHybridModel(CNNTransformerBackbone().to(device), tabular_dim=TB_train.shape[1]).to(device)
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(Y_train)),
                               batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(Y_val)),
                             batch_size=64, shuffle=False)
    optimizer = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, y in train_loader:
            ecg, tab, y = ecg.to(device), tab.to(device), y.to(device)
            optimizer.zero_grad()
            loss = masked_bce_loss(net(ecg, tab), y, pos_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, y in val_loader:
                logits = net(ecg.to(device), tab.to(device))
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[hybrid] Epoch {epoch}: val_HFrEF_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[hybrid] Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, '/kaggle/working/hybrid_model.pt')
    print(f"[hybrid] Saved. Best val AUROC: {best_val_auroc:.4f}")
    return net

net_hybrid = train_hybrid(seed=1)

[hybrid] Epoch 0: val_HFrEF_AUROC=0.8311
[hybrid] Epoch 1: val_HFrEF_AUROC=0.8431
[hybrid] Epoch 2: val_HFrEF_AUROC=0.8433
[hybrid] Epoch 3: val_HFrEF_AUROC=0.8463
[hybrid] Epoch 4: val_HFrEF_AUROC=0.8414
[hybrid] Epoch 5: val_HFrEF_AUROC=0.8475
[hybrid] Epoch 6: val_HFrEF_AUROC=0.8402
[hybrid] Epoch 7: val_HFrEF_AUROC=0.8420
[hybrid] Epoch 8: val_HFrEF_AUROC=0.8418
[hybrid] Epoch 9: val_HFrEF_AUROC=0.8411
[hybrid] Epoch 10: val_HFrEF_AUROC=0.8378
[hybrid] Early stopping at epoch 10
[hybrid] Saved. Best val AUROC: 0.8475


In [10]:
# Cell 7: hybrid final test evaluation
net_hybrid.eval()
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(Y_test)), batch_size=64)
all_logits, all_y = [], []
with torch.no_grad():
    for ecg, tab, y in test_loader:
        logits = net_hybrid(ecg.to(device), tab.to(device))
        all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
probs = 1 / (1 + np.exp(-all_logits))

print("=== HYBRID FINAL TEST RESULTS ===")
for i, name in enumerate(['hfref', 'rv_dysf', 'lvh', 'shd']):
    m = ~np.isnan(all_y[:, i])
    yt, yp = all_y[m, i], probs[m, i]
    auroc = roc_auc_score(yt, yp)
    auprc = average_precision_score(yt, yp)
    pred = (yp > 0.5).astype(int)
    acc = accuracy_score(yt, pred)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"  [{name}] AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")

=== HYBRID FINAL TEST RESULTS ===
  [hfref] AUROC=0.8510 AUPRC=0.4757 acc=0.7808 sens=0.7479 spec=0.7855
  [rv_dysf] AUROC=0.8389 AUPRC=0.3222 acc=0.8032 sens=0.7041 spec=0.8115
  [lvh] AUROC=0.7166 AUPRC=0.3564 acc=0.6911 sens=0.5966 spec=0.7140
  [shd] AUROC=0.7933 AUPRC=0.7445 acc=0.7290 sens=0.6281 spec=0.8038


In [11]:
# Cell 8: hybrid training with data augmentation + stronger regularization (Lever 1)

def augment_ecg(x, training=True):
    """Light, physiologically-plausible augmentation: only applied during training."""
    if not training:
        return x
    B = x.shape[0]
    # random small time shift (+/- 20 samples out of 2500)
    shift = torch.randint(-20, 21, (1,)).item()
    x = torch.roll(x, shifts=shift, dims=1)
    # random amplitude scaling (0.9x - 1.1x)
    scale = 0.9 + 0.2 * torch.rand(B, 1, device=x.device)
    x = x * scale
    # light Gaussian noise
    noise = torch.randn_like(x) * 0.02
    x = x + noise
    return x

class SingleLeadHybridModelV2(nn.Module):
    """Same as before but with higher dropout throughout."""
    def __init__(self, backbone, tabular_dim, feat_dim=256, hidden=128, n_targets=4):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        self.head_trunk = nn.Sequential(nn.Linear(feat_dim + 32, hidden), nn.ReLU(), nn.Dropout(0.35))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, ecg, tabular):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        fused = torch.cat([feats, tab], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

def train_hybrid_v2(seed=1, epochs=20, patience=6):
    torch.manual_seed(seed)
    # add dropout inside the transformer layers too, for stronger regularization
    backbone = CNNTransformerBackbone(n_layers=3)
    for m in backbone.transformer.layers:
        m.dropout.p = 0.25
    net = SingleLeadHybridModelV2(backbone.to(device), tabular_dim=TB_train.shape[1]).to(device)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(Y_train)),
                               batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(Y_val)),
                             batch_size=64, shuffle=False)
    optimizer = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=5e-4)  # stronger weight decay
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, y in train_loader:
            ecg, tab, y = ecg.to(device), tab.to(device), y.to(device)
            ecg = augment_ecg(ecg, training=True)   # <-- augmentation applied here, training only
            optimizer.zero_grad()
            loss = masked_bce_loss(net(ecg, tab), y, pos_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, y in val_loader:
                logits = net(ecg.to(device), tab.to(device))  # no augmentation at eval
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[hybrid-v2] Epoch {epoch}: val_HFrEF_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[hybrid-v2] Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, '/kaggle/working/hybrid_v2.pt')
    print(f"[hybrid-v2] Saved. Best val AUROC: {best_val_auroc:.4f}")
    return net

net_hybrid_v2 = train_hybrid_v2(seed=1)

[hybrid-v2] Epoch 0: val_HFrEF_AUROC=0.8180
[hybrid-v2] Epoch 1: val_HFrEF_AUROC=0.8344
[hybrid-v2] Epoch 2: val_HFrEF_AUROC=0.8482
[hybrid-v2] Epoch 3: val_HFrEF_AUROC=0.8438
[hybrid-v2] Epoch 4: val_HFrEF_AUROC=0.8482
[hybrid-v2] Epoch 5: val_HFrEF_AUROC=0.8467
[hybrid-v2] Epoch 6: val_HFrEF_AUROC=0.8457
[hybrid-v2] Epoch 7: val_HFrEF_AUROC=0.8365
[hybrid-v2] Epoch 8: val_HFrEF_AUROC=0.8490
[hybrid-v2] Epoch 9: val_HFrEF_AUROC=0.8458
[hybrid-v2] Epoch 10: val_HFrEF_AUROC=0.8440
[hybrid-v2] Epoch 11: val_HFrEF_AUROC=0.8530
[hybrid-v2] Epoch 12: val_HFrEF_AUROC=0.8457
[hybrid-v2] Epoch 13: val_HFrEF_AUROC=0.8500
[hybrid-v2] Epoch 14: val_HFrEF_AUROC=0.8480
[hybrid-v2] Epoch 15: val_HFrEF_AUROC=0.8473
[hybrid-v2] Epoch 16: val_HFrEF_AUROC=0.8467
[hybrid-v2] Epoch 17: val_HFrEF_AUROC=0.8457
[hybrid-v2] Early stopping at epoch 17
[hybrid-v2] Saved. Best val AUROC: 0.8530


In [15]:
# Cell 9: hybrid-v2 final test evaluation
net_hybrid_v2.eval()
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(Y_test)), batch_size=64)
all_logits, all_y = [], []
with torch.no_grad():
    for ecg, tab, y in test_loader:
        logits = net_hybrid_v2(ecg.to(device), tab.to(device))
        all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
probs = 1 / (1 + np.exp(-all_logits))

print("=== HYBRID-V2 (augmented) FINAL TEST RESULTS ===")
for i, name in enumerate(['hfref', 'rv_dysf', 'lvh', 'shd']):
    m = ~np.isnan(all_y[:, i])
    yt, yp = all_y[m, i], probs[m, i]
    auroc = roc_auc_score(yt, yp)
    auprc = average_precision_score(yt, yp)
    pred = (yp > 0.5).astype(int)
    acc = accuracy_score(yt, pred)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"  [{name}] AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")

=== HYBRID-V2 (augmented) FINAL TEST RESULTS ===
  [hfref] AUROC=0.8558 AUPRC=0.4988 acc=0.8061 sens=0.7045 spec=0.8205
  [rv_dysf] AUROC=0.8289 AUPRC=0.3372 acc=0.8003 sens=0.7112 spec=0.8077
  [lvh] AUROC=0.7120 AUPRC=0.3550 acc=0.7222 sens=0.4844 spec=0.7797
  [shd] AUROC=0.7934 AUPRC=0.7517 acc=0.7290 sens=0.5526 spec=0.8598


In [18]:
# Cell 10 (FIXED): compute_signature_features with safe boundary handling
def compute_signature_features(x_leadII):
    x = (x_leadII - x_leadII.mean()) / (x_leadII.std() + 1e-6)
    t = np.linspace(0, 1, len(x))

    path_t_II = np.stack([t, x], axis=1)
    feat_t_II = isig.logsig(path_t_II, PREP)

    lag = np.roll(x, 2); lag[:2] = x[0]
    path_leadlag = np.stack([t, x, lag], axis=1)
    feat_leadlag = isig.logsig(path_leadlag, PREP3)

    # SAFE QRS region: clamp indices to valid bounds
    peak_idx = np.argmax(x)
    lo = max(0, peak_idx - 10)
    hi = min(len(x), peak_idx + 10)
    qrs_region = x[lo:hi]
    if len(qrs_region) == 0:  # extra safety net, should never trigger now
        qrs_region = x

    scalars = np.array([x.max(), x.min(), x.std(), qrs_region.max() - qrs_region.min()])

    return np.concatenate([feat_t_II, feat_leadlag, scalars])

# re-test on the first ECG to confirm the fix works
test_feat = compute_signature_features(X_train[0])
print("Signature feature vector shape:", test_feat.shape)
print("Sample values:", test_feat[:6])

Signature feature vector shape: (23,)
Sample values: [1.         0.         0.08326668 0.00403866 0.49612616 1.        ]


In [19]:
# Cell 11: extract 23 signature features for ALL train/val/test ECGs (using fixed function)
import time

def extract_all(X, tag):
    t0 = time.time()
    feats = np.zeros((len(X), 23), dtype=np.float32)
    for i in range(len(X)):
        feats[i] = compute_signature_features(X[i])
        if i % 10000 == 0:
            print(f"  [{tag}] {i}/{len(X)}")
    print(f"[{tag}] done in {time.time()-t0:.1f}s")
    return feats

sig_train = extract_all(X_train, "train")
sig_val = extract_all(X_val, "val")
sig_test = extract_all(X_test, "test")

print("Shapes:", sig_train.shape, sig_val.shape, sig_test.shape)
print("NaN check:", np.isnan(sig_train).sum(), np.isnan(sig_val).sum(), np.isnan(sig_test).sum())

np.save('/kaggle/working/sig_train.npy', sig_train)
np.save('/kaggle/working/sig_val.npy', sig_val)
np.save('/kaggle/working/sig_test.npy', sig_test)
print("Saved signature features to /kaggle/working/")

  [train] 0/72475
  [train] 10000/72475
  [train] 20000/72475
  [train] 30000/72475
  [train] 40000/72475
  [train] 50000/72475
  [train] 60000/72475
  [train] 70000/72475
[train] done in 31.5s
  [val] 0/4626
[val] done in 2.0s
  [test] 0/5442
[test] done in 2.3s
Shapes: (72475, 23) (4626, 23) (5442, 23)
NaN check: 0 0 0
Saved signature features to /kaggle/working/


In [3]:
!pip install iisignature -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [4]:
# FULL RECOVERY CELL — re-establishes everything needed to run Cell 13 (hybrid+signature training)

# --- Imports ---
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, json, copy, time
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, confusion_matrix
import iisignature as isig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# --- Data loading ---
DATA = '/kaggle/input/datasets/boransaksham/echonext-singlelead'
def load_split(split):
    X = np.load(f'{DATA}/leadII_{split}.npy').astype(np.float32)
    TB = np.load(f'{DATA}/T_B_{split}.npy').astype(np.float32)
    Y = np.load(f'{DATA}/Y_{split}.npy').astype(np.float32)
    return X, TB, Y

X_train, TB_train, Y_train = load_split('train')
X_val, TB_val, Y_val = load_split('val')
X_test, TB_test, Y_test = load_split('test')

def znorm(x):
    return (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-6)
X_train, X_val, X_test = znorm(X_train), znorm(X_val), znorm(X_test)

pos_rates = torch.tensor([0.1789, 0.1324, 0.2438, 0.5237])
pos_weights = ((1 - pos_rates) / pos_rates).to(device)

# --- Reload already-computed signature features ---
sig_train = np.load('/kaggle/working/sig_train.npy')
sig_val = np.load('/kaggle/working/sig_val.npy')
sig_test = np.load('/kaggle/working/sig_test.npy')

print("Data loaded:", X_train.shape, sig_train.shape)

# --- Loss function ---
def masked_bce_loss(logits, targets, pos_weights):
    losses = []
    for i in range(targets.shape[1]):
        col_t, col_l = targets[:, i], logits[:, i]
        mask = ~torch.isnan(col_t)
        if mask.sum() == 0: continue
        losses.append(F.binary_cross_entropy_with_logits(col_l[mask], col_t[mask], pos_weight=pos_weights[i]))
    return torch.stack(losses).mean()

# --- CNN-Transformer hybrid backbone ---
class CNNTransformerBackbone(nn.Module):
    def __init__(self, d_model=128, n_heads=4, n_layers=3, out_dim=256):
        super().__init__()
        self.cnn_stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=7, stride=4, padding=3), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, 500, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                            dropout=0.15, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out_proj = nn.Linear(d_model, out_dim)
    def forward(self, x):
        feats = self.cnn_stem(x.unsqueeze(1))
        feats = feats.transpose(1, 2)
        T = feats.shape[1]
        feats = feats + self.pos_embed[:, :T, :]
        out = self.transformer(feats)
        return self.out_proj(out.mean(dim=1))

# --- Augmentation (Lever 1) ---
def augment_ecg(x, training=True):
    if not training:
        return x
    B = x.shape[0]
    shift = torch.randint(-20, 21, (1,)).item()
    x = torch.roll(x, shifts=shift, dims=1)
    scale = 0.9 + 0.2 * torch.rand(B, 1, device=x.device)
    x = x * scale
    noise = torch.randn_like(x) * 0.02
    x = x + noise
    return x

# --- Signature feature extractor (in case needed again) ---
DEPTH = 3
PREP = isig.prepare(2, DEPTH)
PREP3 = isig.prepare(3, DEPTH)

def compute_signature_features(x_leadII):
    x = (x_leadII - x_leadII.mean()) / (x_leadII.std() + 1e-6)
    t = np.linspace(0, 1, len(x))
    path_t_II = np.stack([t, x], axis=1)
    feat_t_II = isig.logsig(path_t_II, PREP)
    lag = np.roll(x, 2); lag[:2] = x[0]
    path_leadlag = np.stack([t, x, lag], axis=1)
    feat_leadlag = isig.logsig(path_leadlag, PREP3)
    peak_idx = np.argmax(x)
    lo = max(0, peak_idx - 10)
    hi = min(len(x), peak_idx + 10)
    qrs_region = x[lo:hi]
    if len(qrs_region) == 0:
        qrs_region = x
    scalars = np.array([x.max(), x.min(), x.std(), qrs_region.max() - qrs_region.min()])
    return np.concatenate([feat_t_II, feat_leadlag, scalars])

# --- Fusion model: hybrid backbone + tabular + signatures (Lever 2) ---
class SingleLeadHybridSigModel(nn.Module):
    def __init__(self, backbone, tabular_dim, sig_dim=23, feat_dim=256, hidden=128, n_targets=4):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        self.sig_mlp = nn.Sequential(nn.Linear(sig_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        fused_dim = feat_dim + 32 + 32
        self.head_trunk = nn.Sequential(nn.Linear(fused_dim, hidden), nn.ReLU(), nn.Dropout(0.35))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, ecg, tabular, sig):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        s = self.sig_mlp(sig)
        fused = torch.cat([feats, tab, s], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

print("\nAll classes and functions restored. Ready to run training.")

Device: cuda
Data loaded: (72475, 2500) (72475, 23)

All classes and functions restored. Ready to run training.


In [5]:
#CELL 13
def train_hybrid_sig(seed=1, epochs=20, patience=6):
    torch.manual_seed(seed)
    backbone = CNNTransformerBackbone(n_layers=3)
    for m in backbone.transformer.layers:
        m.dropout.p = 0.25
    net = SingleLeadHybridSigModel(backbone.to(device), tabular_dim=TB_train.shape[1]).to(device)

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(sig_train), torch.tensor(Y_train))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(sig_val), torch.tensor(Y_val))
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

    optimizer = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, sig, y in train_loader:
            ecg, tab, sig, y = ecg.to(device), tab.to(device), sig.to(device), y.to(device)
            ecg = augment_ecg(ecg, training=True)
            optimizer.zero_grad()
            loss = masked_bce_loss(net(ecg, tab, sig), y, pos_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, sig, y in val_loader:
                logits = net(ecg.to(device), tab.to(device), sig.to(device))
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[hybrid-sig] Epoch {epoch}: val_HFrEF_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[hybrid-sig] Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, '/kaggle/working/hybrid_sig_model.pt')
    print(f"[hybrid-sig] Saved. Best val AUROC: {best_val_auroc:.4f}")
    return net

net_hybrid_sig = train_hybrid_sig(seed=1)

[hybrid-sig] Epoch 0: val_HFrEF_AUROC=0.8239
[hybrid-sig] Epoch 1: val_HFrEF_AUROC=0.8357
[hybrid-sig] Epoch 2: val_HFrEF_AUROC=0.8330
[hybrid-sig] Epoch 3: val_HFrEF_AUROC=0.8458
[hybrid-sig] Epoch 4: val_HFrEF_AUROC=0.8397
[hybrid-sig] Epoch 5: val_HFrEF_AUROC=0.8468
[hybrid-sig] Epoch 6: val_HFrEF_AUROC=0.8539
[hybrid-sig] Epoch 7: val_HFrEF_AUROC=0.8507
[hybrid-sig] Epoch 8: val_HFrEF_AUROC=0.8525
[hybrid-sig] Epoch 9: val_HFrEF_AUROC=0.8557
[hybrid-sig] Epoch 10: val_HFrEF_AUROC=0.8547
[hybrid-sig] Epoch 11: val_HFrEF_AUROC=0.8497
[hybrid-sig] Epoch 12: val_HFrEF_AUROC=0.8470
[hybrid-sig] Epoch 13: val_HFrEF_AUROC=0.8475
[hybrid-sig] Epoch 14: val_HFrEF_AUROC=0.8472
[hybrid-sig] Epoch 15: val_HFrEF_AUROC=0.8392
[hybrid-sig] Early stopping at epoch 15
[hybrid-sig] Saved. Best val AUROC: 0.8557


In [6]:
# Cell 14: hybrid-sig final test evaluation
net_hybrid_sig.eval()
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(sig_test), torch.tensor(Y_test))
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
all_logits, all_y = [], []
with torch.no_grad():
    for ecg, tab, sig, y in test_loader:
        logits = net_hybrid_sig(ecg.to(device), tab.to(device), sig.to(device))
        all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
probs = 1 / (1 + np.exp(-all_logits))

print("=== HYBRID-SIG (Lever 1 + Lever 2) FINAL TEST RESULTS ===")
for i, name in enumerate(['hfref', 'rv_dysf', 'lvh', 'shd']):
    m = ~np.isnan(all_y[:, i])
    yt, yp = all_y[m, i], probs[m, i]
    auroc = roc_auc_score(yt, yp)
    auprc = average_precision_score(yt, yp)
    pred = (yp > 0.5).astype(int)
    acc = accuracy_score(yt, pred)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"  [{name}] AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")

=== HYBRID-SIG (Lever 1 + Lever 2) FINAL TEST RESULTS ===
  [hfref] AUROC=0.8518 AUPRC=0.4784 acc=0.7721 sens=0.7780 spec=0.7713
  [rv_dysf] AUROC=0.8300 AUPRC=0.3063 acc=0.7519 sens=0.7733 spec=0.7501
  [lvh] AUROC=0.7099 AUPRC=0.3476 acc=0.6439 sens=0.6937 spec=0.6318
  [shd] AUROC=0.7874 AUPRC=0.7444 acc=0.7172 sens=0.6536 spec=0.7644


In [8]:
# Recovery addition: restore Lever 1 pieces (SingleLeadHybridModelV2 + train_hybrid_v2)

class SingleLeadHybridModelV2(nn.Module):
    def __init__(self, backbone, tabular_dim, feat_dim=256, hidden=128, n_targets=4):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        self.head_trunk = nn.Sequential(nn.Linear(feat_dim + 32, hidden), nn.ReLU(), nn.Dropout(0.35))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, ecg, tabular):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        fused = torch.cat([feats, tab], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

def train_hybrid_v2(seed=1, epochs=20, patience=6):
    torch.manual_seed(seed)
    backbone = CNNTransformerBackbone(n_layers=3)
    for m in backbone.transformer.layers:
        m.dropout.p = 0.25
    net = SingleLeadHybridModelV2(backbone.to(device), tabular_dim=TB_train.shape[1]).to(device)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(Y_train)),
                               batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(Y_val)),
                             batch_size=64, shuffle=False)
    optimizer = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, y in train_loader:
            ecg, tab, y = ecg.to(device), tab.to(device), y.to(device)
            ecg = augment_ecg(ecg, training=True)
            optimizer.zero_grad()
            loss = masked_bce_loss(net(ecg, tab), y, pos_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, y in val_loader:
                logits = net(ecg.to(device), tab.to(device))
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[hybrid-v2 seed{seed}] Epoch {epoch}: val_HFrEF_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[hybrid-v2 seed{seed}] Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/hybrid_v2_seed{seed}.pt')
    print(f"[hybrid-v2 seed{seed}] Saved. Best val AUROC: {best_val_auroc:.4f}")
    return net

print("Lever 1 pieces restored: SingleLeadHybridModelV2, train_hybrid_v2")

Lever 1 pieces restored: SingleLeadHybridModelV2, train_hybrid_v2


In [9]:
# Cell 15: train Hybrid-V2 with 2 more seeds for ensembling
net_v2_seed2 = train_hybrid_v2(seed=2)
net_v2_seed3 = train_hybrid_v2(seed=3)

KeyboardInterrupt: 

In [3]:
# Cell 16: ensemble all 3 seeds, final test evaluation
def get_test_probs_v2(net):
    net.eval()
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(Y_test)), batch_size=64)
    all_logits, all_y = [], []
    with torch.no_grad():
        for ecg, tab, y in test_loader:
            logits = net(ecg.to(device), tab.to(device))
            all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

logits1, y_test = get_test_probs_v2(net_v2_seed1)
logits2, _ = get_test_probs_v2(net_v2_seed2)
logits3, _ = get_test_probs_v2(net_v2_seed3)

probs1 = 1/(1+np.exp(-logits1))
probs2 = 1/(1+np.exp(-logits2))
probs3 = 1/(1+np.exp(-logits3))
probs_ensemble = (probs1 + probs2 + probs3) / 3

print("=== INDIVIDUAL SEEDS (hfref test AUROC) ===")
m = ~np.isnan(y_test[:, 0])
print(f"  seed1: {roc_auc_score(y_test[m,0], probs1[m,0]):.4f}")
print(f"  seed2: {roc_auc_score(y_test[m,0], probs2[m,0]):.4f}")
print(f"  seed3: {roc_auc_score(y_test[m,0], probs3[m,0]):.4f}")

print("\n=== 3-SEED ENSEMBLE FINAL TEST RESULTS ===")
for i, name in enumerate(['hfref', 'rv_dysf', 'lvh', 'shd']):
    mm = ~np.isnan(y_test[:, i])
    yt, yp = y_test[mm, i], probs_ensemble[mm, i]
    auroc = roc_auc_score(yt, yp)
    auprc = average_precision_score(yt, yp)
    pred = (yp > 0.5).astype(int)
    acc = accuracy_score(yt, pred)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"  [{name}] AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")

NameError: name 'net_v2_seed1' is not defined

In [10]:
# Cell 15 (updated): train all 3 seeds fresh, including seed=1 again
net_v2_seed1 = train_hybrid_v2(seed=1)
net_v2_seed2 = train_hybrid_v2(seed=2)
net_v2_seed3 = train_hybrid_v2(seed=3)

[hybrid-v2 seed1] Epoch 0: val_HFrEF_AUROC=0.8196
[hybrid-v2 seed1] Epoch 1: val_HFrEF_AUROC=0.8343
[hybrid-v2 seed1] Epoch 2: val_HFrEF_AUROC=0.8483
[hybrid-v2 seed1] Epoch 3: val_HFrEF_AUROC=0.8444
[hybrid-v2 seed1] Epoch 4: val_HFrEF_AUROC=0.8480
[hybrid-v2 seed1] Epoch 5: val_HFrEF_AUROC=0.8469
[hybrid-v2 seed1] Epoch 6: val_HFrEF_AUROC=0.8467
[hybrid-v2 seed1] Epoch 7: val_HFrEF_AUROC=0.8404
[hybrid-v2 seed1] Epoch 8: val_HFrEF_AUROC=0.8523
[hybrid-v2 seed1] Epoch 9: val_HFrEF_AUROC=0.8477
[hybrid-v2 seed1] Epoch 10: val_HFrEF_AUROC=0.8446
[hybrid-v2 seed1] Epoch 11: val_HFrEF_AUROC=0.8544
[hybrid-v2 seed1] Epoch 12: val_HFrEF_AUROC=0.8489
[hybrid-v2 seed1] Epoch 13: val_HFrEF_AUROC=0.8504
[hybrid-v2 seed1] Epoch 14: val_HFrEF_AUROC=0.8477
[hybrid-v2 seed1] Epoch 15: val_HFrEF_AUROC=0.8451
[hybrid-v2 seed1] Epoch 16: val_HFrEF_AUROC=0.8477
[hybrid-v2 seed1] Epoch 17: val_HFrEF_AUROC=0.8467
[hybrid-v2 seed1] Early stopping at epoch 17
[hybrid-v2 seed1] Saved. Best val AUROC: 0.8544

In [ ]:
!pip install iisignature -q
print("Done installing.")

In [ ]:
!pip install --no-cache-dir --no-build-isolation iisignature -q 2>&1 | tail -20
print("Install attempt finished.")

In [4]:
!pip install iisignature -q

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, json, copy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, confusion_matrix
import iisignature as isig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

DATA = '/kaggle/input/datasets/boransaksham/echonext-singlelead'
def load_split(split):
    X = np.load(f'{DATA}/leadII_{split}.npy').astype(np.float32)
    TB = np.load(f'{DATA}/T_B_{split}.npy').astype(np.float32)
    Y = np.load(f'{DATA}/Y_{split}.npy').astype(np.float32)
    return X, TB, Y

X_train, TB_train, Y_train = load_split('train')
X_val, TB_val, Y_val = load_split('val')
X_test, TB_test, Y_test = load_split('test')

def znorm(x):
    return (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-6)
X_train, X_val, X_test = znorm(X_train), znorm(X_val), znorm(X_test)

# reload already-computed signature features (no need to recompute)
sig_train = np.load('/kaggle/working/sig_train.npy')
sig_val = np.load('/kaggle/working/sig_val.npy')
sig_test = np.load('/kaggle/working/sig_test.npy')
print("Data loaded:", X_train.shape, sig_train.shape)

class CNNTransformerBackbone(nn.Module):
    def __init__(self, d_model=128, n_heads=4, n_layers=3, out_dim=256):
        super().__init__()
        self.cnn_stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=7, stride=4, padding=3), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, 500, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                            dropout=0.15, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out_proj = nn.Linear(d_model, out_dim)
    def forward(self, x):
        feats = self.cnn_stem(x.unsqueeze(1))
        feats = feats.transpose(1, 2)
        T = feats.shape[1]
        feats = feats + self.pos_embed[:, :T, :]
        out = self.transformer(feats)
        return self.out_proj(out.mean(dim=1))

class SingleLeadHybridSigModel(nn.Module):
    def __init__(self, backbone, tabular_dim, sig_dim=23, feat_dim=256, hidden=128, n_targets=4):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        self.sig_mlp = nn.Sequential(nn.Linear(sig_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        fused_dim = feat_dim + 32 + 32
        self.head_trunk = nn.Sequential(nn.Linear(fused_dim, hidden), nn.ReLU(), nn.Dropout(0.35))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, ecg, tabular, sig):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        s = self.sig_mlp(sig)
        fused = torch.cat([feats, tab, s], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

# rebuild architecture, then load the SAVED weights (no retraining needed)
net_hybrid_sig = SingleLeadHybridSigModel(
    CNNTransformerBackbone(n_layers=3).to(device),
    tabular_dim=TB_train.shape[1]
).to(device)
net_hybrid_sig.load_state_dict(torch.load('/kaggle/working/hybrid_sig_model.pt'))
net_hybrid_sig.eval()
print("net_hybrid_sig reloaded successfully from saved checkpoint.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
Device: cuda
Data loaded: (72475, 2500) (72475, 23)
net_hybrid_sig reloaded successfully from saved checkpoint.


In [5]:
def get_probs(net, X, TB, sig, Y):
    net.eval()
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(TB), torch.tensor(sig), torch.tensor(Y)), batch_size=64)
    all_logits, all_y = [], []
    with torch.no_grad():
        for ecg, tab, s, y in loader:
            logits = net(ecg.to(device), tab.to(device), s.to(device))
            all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

val_logits, y_val_out = get_probs(net_hybrid_sig, X_val, TB_val, sig_val, Y_val)
val_probs = 1/(1+np.exp(-val_logits))
m_val = ~np.isnan(y_val_out[:, 0])
yv, pv = y_val_out[m_val, 0], val_probs[m_val, 0]

thresholds = np.linspace(0.01, 0.99, 500)
best_thresh, best_diff = 0.5, 999
for t in thresholds:
    pred = (pv > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, pred).ravel()
    sens = tp / (tp + fn) if (tp+fn) > 0 else 0
    diff = abs(sens - 0.70)
    if diff < best_diff:
        best_diff, best_thresh = diff, t
print(f"Threshold for ~70% sensitivity: {best_thresh:.3f}")

test_logits, y_test_sig = get_probs(net_hybrid_sig, X_test, TB_test, sig_test, Y_test)
test_probs = 1/(1+np.exp(-test_logits))
mm = ~np.isnan(y_test_sig[:, 0])
yt, pt = y_test_sig[mm, 0], test_probs[mm, 0]
test_pred = (pt > best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(yt, test_pred).ravel()

sensitivity, specificity = tp/(tp+fn), tn/(tn+fp)
ppv = tp/(tp+fp) if (tp+fp)>0 else 0
npv = tn/(tn+fn) if (tn+fn)>0 else 0
accuracy = (tp+tn)/(tp+tn+fp+fn)

print(f"\n=== SIGNATURE MODEL, TEST at {best_thresh:.3f} threshold (70% sens match) ===")
print(f"Sensitivity: {sensitivity*100:.1f}%  Specificity: {specificity*100:.1f}%")
print(f"PPV: {ppv*100:.1f}%  NPV: {npv*100:.1f}%  Accuracy: {accuracy*100:.1f}%")

Threshold for ~70% sensitivity: 0.566

=== SIGNATURE MODEL, TEST at 0.566 threshold (70% sens match) ===
Sensitivity: 71.0%  Specificity: 82.4%
PPV: 36.3%  NPV: 95.2%  Accuracy: 81.0%


In [6]:
n_flip_train = sum(1 for x in X_train if abs(x.min()) > x.max())
n_flip_val = sum(1 for x in X_val if abs(x.min()) > x.max())
n_flip_test = sum(1 for x in X_test if abs(x.min()) > x.max())
print(f"Train: {n_flip_train}/{len(X_train)} ({100*n_flip_train/len(X_train):.2f}%)")
print(f"Val:   {n_flip_val}/{len(X_val)} ({100*n_flip_val/len(X_val):.2f}%)")
print(f"Test:  {n_flip_test}/{len(X_test)} ({100*n_flip_test/len(X_test):.2f}%)")

Train: 14051/72475 (19.39%)
Val:   823/4626 (17.79%)
Test:  955/5442 (17.55%)


In [8]:
flip_mask_test = np.array([abs(x.min()) > x.max() for x in X_test])
print("HFrEF rate among 'flipped' ECGs:", Y_test[flip_mask_test, 0].mean())
print("HFrEF rate among 'normal' ECGs:", Y_test[~flip_mask_test, 0].mean())

HFrEF rate among 'flipped' ECGs: nan
HFrEF rate among 'normal' ECGs: nan


In [9]:
# Exclude NaN-masked HFrEF labels before computing the rates
valid_mask = ~np.isnan(Y_test[:, 0])
flip_mask_test = np.array([abs(x.min()) > x.max() for x in X_test])

flip_valid = flip_mask_test & valid_mask
normal_valid = (~flip_mask_test) & valid_mask

print("Flipped ECGs (valid labels):", flip_valid.sum())
print("Normal ECGs (valid labels):", normal_valid.sum())
print("HFrEF rate among 'flipped' ECGs:", Y_test[flip_valid, 0].mean())
print("HFrEF rate among 'normal' ECGs:", Y_test[normal_valid, 0].mean())

Flipped ECGs (valid labels): 891
Normal ECGs (valid labels): 3936
HFrEF rate among 'flipped' ECGs: 0.24803591
HFrEF rate among 'normal' ECGs: 0.09603658


In [10]:
class SingleTargetHFModel(nn.Module):
    def __init__(self, backbone, tabular_dim, feat_dim=256, hidden=128):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        self.head_trunk = nn.Sequential(nn.Linear(feat_dim + 32, hidden), nn.ReLU(), nn.Dropout(0.35))
        self.head = nn.Linear(hidden, 1)   # ONE head only
    def forward(self, ecg, tabular):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        fused = torch.cat([feats, tab], dim=1)
        h = self.head_trunk(fused)
        return self.head(h)

def augment_ecg(x, training=True):
    if not training: return x
    B = x.shape[0]
    shift = torch.randint(-20, 21, (1,)).item()
    x = torch.roll(x, shifts=shift, dims=1)
    scale = 0.9 + 0.2 * torch.rand(B, 1, device=x.device)
    x = x * scale
    x = x + torch.randn_like(x) * 0.02
    return x

pos_rates = torch.tensor([0.1789, 0.1324, 0.2438, 0.5237])
pos_weights = ((1 - pos_rates) / pos_rates).to(device)

def train_single_target(seed=1, epochs=20, patience=6):
    torch.manual_seed(seed)
    backbone = CNNTransformerBackbone(n_layers=3)
    for m in backbone.transformer.layers:
        m.dropout.p = 0.25
    net = SingleTargetHFModel(backbone.to(device), tabular_dim=TB_train.shape[1]).to(device)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(Y_train[:, 0:1])),
                               batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(Y_val[:, 0:1])),
                             batch_size=64, shuffle=False)

    pos_weight_hfref = pos_weights[0:1]
    optimizer = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, y in train_loader:
            ecg, tab, y = ecg.to(device), tab.to(device), y.to(device)
            ecg = augment_ecg(ecg, training=True)
            mask = ~torch.isnan(y[:, 0])
            if mask.sum() == 0: continue
            optimizer.zero_grad()
            logits = net(ecg, tab)
            loss = F.binary_cross_entropy_with_logits(logits[mask, 0], y[mask, 0], pos_weight=pos_weight_hfref[0])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, y in val_loader:
                logits = net(ecg.to(device), tab.to(device))
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[single-target] Epoch {epoch}: val_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, '/kaggle/working/single_target_hfref.pt')
    print(f"Saved. Best val AUROC: {best_val_auroc:.4f}")
    return net

net_single = train_single_target(seed=1)

[single-target] Epoch 0: val_AUROC=0.8271
[single-target] Epoch 1: val_AUROC=0.8279
[single-target] Epoch 2: val_AUROC=0.8317
[single-target] Epoch 3: val_AUROC=0.8429
[single-target] Epoch 4: val_AUROC=0.8477
[single-target] Epoch 5: val_AUROC=0.8464
[single-target] Epoch 6: val_AUROC=0.8495
[single-target] Epoch 7: val_AUROC=0.8514
[single-target] Epoch 8: val_AUROC=0.8497
[single-target] Epoch 9: val_AUROC=0.8542
[single-target] Epoch 10: val_AUROC=0.8504
[single-target] Epoch 11: val_AUROC=0.8490
[single-target] Epoch 12: val_AUROC=0.8541
[single-target] Epoch 13: val_AUROC=0.8463
[single-target] Epoch 14: val_AUROC=0.8387
[single-target] Epoch 15: val_AUROC=0.8416
Early stopping at epoch 15
Saved. Best val AUROC: 0.8542


In [11]:
net_single.eval()
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(Y_test[:, 0:1])), batch_size=64)
all_logits, all_y = [], []
with torch.no_grad():
    for ecg, tab, y in test_loader:
        logits = net_single(ecg.to(device), tab.to(device))
        all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
probs = 1/(1+np.exp(-all_logits))
m = ~np.isnan(all_y[:, 0])
auroc = roc_auc_score(all_y[m, 0], probs[m, 0])
auprc = average_precision_score(all_y[m, 0], probs[m, 0])
pred = (probs[m, 0] > 0.5).astype(int)
acc = accuracy_score(all_y[m, 0], pred)
tn, fp, fn, tp = confusion_matrix(all_y[m, 0], pred).ravel()

print("=== SINGLE-TARGET (HFrEF only) FINAL TEST RESULTS ===")
print(f"AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")
print(f"\nCompare to multi-target Hybrid-V2 test AUROC: 0.8558")

=== SINGLE-TARGET (HFrEF only) FINAL TEST RESULTS ===
AUROC=0.8631 AUPRC=0.4967 acc=0.7725 sens=0.8197 spec=0.7658

Compare to multi-target Hybrid-V2 test AUROC: 0.8558


In [12]:
class SingleTargetHFSigModel(nn.Module):
    def __init__(self, backbone, tabular_dim, sig_dim=23, feat_dim=256, hidden=128):
        super().__init__()
        self.backbone = backbone
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        self.sig_mlp = nn.Sequential(nn.Linear(sig_dim, 32), nn.ReLU(), nn.Dropout(0.25))
        fused_dim = feat_dim + 32 + 32
        self.head_trunk = nn.Sequential(nn.Linear(fused_dim, hidden), nn.ReLU(), nn.Dropout(0.35))
        self.head = nn.Linear(hidden, 1)   # ONE head only
    def forward(self, ecg, tabular, sig):
        feats = self.backbone(ecg)
        tab = self.tabular_mlp(tabular)
        s = self.sig_mlp(sig)
        fused = torch.cat([feats, tab, s], dim=1)
        h = self.head_trunk(fused)
        return self.head(h)

def augment_ecg(x, training=True):
    if not training: return x
    B = x.shape[0]
    shift = torch.randint(-20, 21, (1,)).item()
    x = torch.roll(x, shifts=shift, dims=1)
    scale = 0.9 + 0.2 * torch.rand(B, 1, device=x.device)
    x = x * scale
    x = x + torch.randn_like(x) * 0.02
    return x

pos_rates = torch.tensor([0.1789, 0.1324, 0.2438, 0.5237])
pos_weights = ((1 - pos_rates) / pos_rates).to(device)

def train_single_target_sig(seed=1, epochs=20, patience=6):
    torch.manual_seed(seed)
    backbone = CNNTransformerBackbone(n_layers=3)
    for m in backbone.transformer.layers:
        m.dropout.p = 0.25
    net = SingleTargetHFSigModel(backbone.to(device), tabular_dim=TB_train.shape[1]).to(device)

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(TB_train), torch.tensor(sig_train), torch.tensor(Y_train[:, 0:1]))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(TB_val), torch.tensor(sig_val), torch.tensor(Y_val[:, 0:1]))
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    pos_weight_hfref = pos_weights[0:1]
    optimizer = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auroc, patience_ctr, best_state = -1, 0, None
    for epoch in range(epochs):
        net.train()
        for ecg, tab, sig, y in train_loader:
            ecg, tab, sig, y = ecg.to(device), tab.to(device), sig.to(device), y.to(device)
            ecg = augment_ecg(ecg, training=True)
            mask = ~torch.isnan(y[:, 0])
            if mask.sum() == 0: continue
            optimizer.zero_grad()
            logits = net(ecg, tab, sig)
            loss = F.binary_cross_entropy_with_logits(logits[mask, 0], y[mask, 0], pos_weight=pos_weight_hfref[0])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        net.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for ecg, tab, sig, y in val_loader:
                logits = net(ecg.to(device), tab.to(device), sig.to(device))
                all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
        all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
        m = ~np.isnan(all_y[:, 0])
        val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
        print(f"[single-target-sig] Epoch {epoch}: val_AUROC={val_auroc:.4f}")

        if val_auroc > best_val_auroc:
            best_val_auroc, patience_ctr, best_state = val_auroc, 0, copy.deepcopy(net.state_dict())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}"); break

    net.load_state_dict(best_state)
    torch.save(best_state, '/kaggle/working/single_target_sig_hfref.pt')
    print(f"Saved. Best val AUROC: {best_val_auroc:.4f}")
    return net

net_single_sig = train_single_target_sig(seed=1)

[single-target-sig] Epoch 0: val_AUROC=0.8188
[single-target-sig] Epoch 1: val_AUROC=0.8327
[single-target-sig] Epoch 2: val_AUROC=0.8394
[single-target-sig] Epoch 3: val_AUROC=0.8474
[single-target-sig] Epoch 4: val_AUROC=0.8464
[single-target-sig] Epoch 5: val_AUROC=0.8475
[single-target-sig] Epoch 6: val_AUROC=0.8502
[single-target-sig] Epoch 7: val_AUROC=0.8500
[single-target-sig] Epoch 8: val_AUROC=0.8572
[single-target-sig] Epoch 9: val_AUROC=0.8525
[single-target-sig] Epoch 10: val_AUROC=0.8488
[single-target-sig] Epoch 11: val_AUROC=0.8511
[single-target-sig] Epoch 12: val_AUROC=0.8528
[single-target-sig] Epoch 13: val_AUROC=0.8506
[single-target-sig] Epoch 14: val_AUROC=0.8562
Early stopping at epoch 14
Saved. Best val AUROC: 0.8572


In [13]:
net_single_sig.eval()
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(TB_test), torch.tensor(sig_test), torch.tensor(Y_test[:, 0:1]))
test_loader = DataLoader(test_ds, batch_size=64)
all_logits, all_y = [], []
with torch.no_grad():
    for ecg, tab, sig, y in test_loader:
        logits = net_single_sig(ecg.to(device), tab.to(device), sig.to(device))
        all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
probs = 1/(1+np.exp(-all_logits))
m = ~np.isnan(all_y[:, 0])
auroc = roc_auc_score(all_y[m, 0], probs[m, 0])
auprc = average_precision_score(all_y[m, 0], probs[m, 0])
pred = (probs[m, 0] > 0.5).astype(int)
acc = accuracy_score(all_y[m, 0], pred)
tn, fp, fn, tp = confusion_matrix(all_y[m, 0], pred).ravel()

print("=== SINGLE-TARGET + SIGNATURES FINAL TEST RESULTS ===")
print(f"AUROC={auroc:.4f} AUPRC={auprc:.4f} acc={acc:.4f} sens={tp/(tp+fn):.4f} spec={tn/(tn+fp):.4f}")
print(f"\nCompare to multi-target signature model test AUROC: 0.8518")

=== SINGLE-TARGET + SIGNATURES FINAL TEST RESULTS ===
AUROC=0.8595 AUPRC=0.4970 acc=0.7854 sens=0.7663 spec=0.7881

Compare to multi-target signature model test AUROC: 0.8518
